# E-Commerce Customer Segmentation & Sales Analysis

**Dataset**: Online Retail Dataset (UCI ML Repository)

In this analysis, I'll be exploring e-commerce transaction data to understand customer behavior and identify opportunities for business growth. The main goals are:

- Segment customers using RFM (Recency, Frequency, Monetary) analysis
- Identify sales patterns and trends
- Analyze geographic and product performance
- Build a cohort analysis to understand retention


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# basic setup
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)


## Loading the Data

Using the Online Retail dataset from UCI - it contains about a year of transactions from a UK retailer.


In [ ]:
# load the excel file
df = pd.read_excel('Online_Retail.xlsx')
print(f"Loaded {len(df)} rows")
df.head()


In [ ]:
# quick look at the data types and missing values
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# looks like ~25% of CustomerID is missing - need to drop those for customer analysis
# also some negative quantities which are probably returns

df_clean = df.copy()

# drop missing customer IDs - can't do customer segmentation without them
df_clean = df_clean.dropna(subset=['CustomerID'])

# filter out returns (negative qty) and zero/negative prices
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# add some useful columns
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Year'] = df_clean['InvoiceDate'].dt.year
df_clean['Month'] = df_clean['InvoiceDate'].dt.month
df_clean['Weekday'] = df_clean['InvoiceDate'].dt.day_name()
df_clean['Hour'] = df_clean['InvoiceDate'].dt.hour

print(f"Cleaned data: {len(df_clean)} rows")
print(f"Date range: {df_clean['InvoiceDate'].min().date()} to {df_clean['InvoiceDate'].max().date()}")
print(f"Customers: {df_clean['CustomerID'].nunique()}")


## Quick Business Overview


In [ ]:
# let's see some high level metrics
total_rev = df_clean['TotalAmount'].sum()
num_customers = df_clean['CustomerID'].nunique()
num_orders = df_clean['InvoiceNo'].nunique()
avg_order = total_rev / num_orders

print(f"Total Revenue: £{total_rev:,.0f}")
print(f"Unique Customers: {num_customers:,}")
print(f"Total Orders: {num_orders:,}")
print(f"Avg Order Value: £{avg_order:.2f}")


In [ ]:
# sales trends
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# monthly trend
monthly = df_clean.groupby(df_clean['InvoiceDate'].dt.to_period('M'))['TotalAmount'].sum()
monthly.plot(ax=axes[0,0], color='steelblue', marker='o')
axes[0,0].set_title('Monthly Revenue')
axes[0,0].set_ylabel('Revenue (£)')

# by day of week
weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily = df_clean.groupby('Weekday')['TotalAmount'].sum().reindex(weekday_order)
daily.plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Revenue by Day of Week')
axes[0,1].tick_params(axis='x', rotation=45)

# hourly pattern
hourly = df_clean.groupby('Hour')['TotalAmount'].sum()
hourly.plot(ax=axes[1,0], color='green')
axes[1,0].set_title('Revenue by Hour')
axes[1,0].set_xlabel('Hour of Day')

# top countries
top_countries = df_clean.groupby('Country')['TotalAmount'].sum().nlargest(10)
top_countries.plot(kind='barh', ax=axes[1,1])
axes[1,1].set_title('Top 10 Countries')

plt.tight_layout()
plt.show()


## RFM Customer Segmentation

RFM is a classic approach - looking at how recently someone bought (Recency), how often (Frequency), and how much (Monetary).


In [ ]:
# calculate RFM for each customer
# using day after last transaction as reference point
snapshot_date = df_clean['InvoiceDate'].max() + timedelta(days=1)

rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']

rfm.describe()


In [ ]:
# score each dimension 1-5 using quantiles
# for recency, lower is better so we reverse the labels
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

rfm['RFM_Score'] = rfm['R'].astype(int) + rfm['F'].astype(int) + rfm['M'].astype(int)
rfm.head()


In [ ]:
# create segments based on combined score
def get_segment(score):
    if score >= 13:
        return 'Champions'
    elif score >= 11:
        return 'Loyal'
    elif score >= 9:
        return 'Potential Loyal'
    elif score >= 7:
        return 'New/Promising'
    elif score >= 5:
        return 'At Risk'
    else:
        return 'Lost'

rfm['Segment'] = rfm['RFM_Score'].apply(get_segment)

# see the breakdown
seg_summary = rfm.groupby('Segment').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'sum']
})
seg_summary.columns = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary', 'Total_Revenue']
seg_summary['Count'] = rfm.groupby('Segment').size()
seg_summary['Rev_Pct'] = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)

seg_summary.sort_values('Count', ascending=False)


In [ ]:
# visualize the segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# pie chart of customer distribution
seg_counts = rfm['Segment'].value_counts()
axes[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Customer Distribution by Segment')

# revenue by segment
seg_summary['Total_Revenue'].sort_values().plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Revenue by Segment')
axes[1].set_xlabel('Revenue (£)')

plt.tight_layout()
plt.show()

# key insight
print(f"\nChampions are {seg_summary.loc['Champions', 'Count']/len(rfm)*100:.1f}% of customers but drive {seg_summary.loc['Champions', 'Rev_Pct']}% of revenue")


## Product & Geographic Analysis


In [ ]:
# top products by revenue
products = df_clean.groupby(['StockCode', 'Description']).agg({
    'Quantity': 'sum',
    'TotalAmount': 'sum',
    'CustomerID': 'nunique'
}).sort_values('TotalAmount', ascending=False)
products.columns = ['Qty_Sold', 'Revenue', 'Unique_Buyers']

print("Top 10 Products by Revenue:")
products.head(10)


In [ ]:
# geographic breakdown
geo = df_clean.groupby('Country').agg({
    'TotalAmount': 'sum',
    'CustomerID': 'nunique',
    'InvoiceNo': 'nunique'
})
geo.columns = ['Revenue', 'Customers', 'Orders']
geo['Rev_per_Customer'] = geo['Revenue'] / geo['Customers']
geo = geo.sort_values('Revenue', ascending=False)

print("Top 10 Countries:")
print(geo.head(10))

# UK dominance
uk_pct = geo.loc['United Kingdom', 'Revenue'] / geo['Revenue'].sum() * 100
print(f"\nUK accounts for {uk_pct:.1f}% of total revenue")


## Cohort Retention Analysis

Looking at how well we retain customers over time based on their first purchase month.


In [ ]:
# assign each customer to their first purchase month (cohort)
df_clean['OrderMonth'] = df_clean['InvoiceDate'].dt.to_period('M').astype(str)
first_purchase = df_clean.groupby('CustomerID')['InvoiceDate'].min().dt.to_period('M').astype(str)
df_clean['Cohort'] = df_clean['CustomerID'].map(first_purchase)

# count unique customers per cohort per month
cohort_data = df_clean.groupby(['Cohort', 'OrderMonth'])['CustomerID'].nunique().reset_index()
cohort_sizes = df_clean.groupby('Cohort')['CustomerID'].nunique()

# calculate months since first purchase
cohort_data['Period'] = (pd.to_datetime(cohort_data['OrderMonth']) - 
                         pd.to_datetime(cohort_data['Cohort'])).dt.days // 30

# pivot for heatmap
cohort_data = cohort_data.drop_duplicates(['Cohort', 'Period'])
cohort_pivot = cohort_data.pivot(index='Cohort', columns='Period', values='CustomerID')

# convert to retention rate
for col in cohort_pivot.columns:
    cohort_pivot[col] = cohort_pivot[col] / cohort_sizes

cohort_pivot = cohort_pivot.fillna(0)


In [ ]:
# plot the retention heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(cohort_pivot, annot=True, fmt='.0%', cmap='YlOrRd')
plt.title('Customer Retention by Cohort')
plt.xlabel('Months Since First Purchase')
plt.ylabel('Cohort (First Purchase Month)')
plt.show()

# average retention by month
avg_ret = cohort_pivot.mean()
print("Average Retention Rates:")
for i, rate in avg_ret.items():
    if i <= 6:
        print(f"  Month {i}: {rate:.1%}")


## Key Takeaways


In [ ]:
# pulling together the key findings
print("=" * 50)
print("SUMMARY OF FINDINGS")
print("=" * 50)

print("\n1. CUSTOMER SEGMENTS:")
print(f"   - Champions: {seg_summary.loc['Champions', 'Count']} customers ({seg_summary.loc['Champions', 'Rev_Pct']}% of revenue)")
print(f"   - At Risk: {seg_summary.loc['At Risk', 'Count']} customers need attention")
print(f"   - Lost: {seg_summary.loc['Lost', 'Count']} customers to potentially win back")

print("\n2. SALES PATTERNS:")
print(f"   - Peak month: {monthly.idxmax()}")
print(f"   - Best day: {daily.idxmax()}")
print(f"   - Peak hour: {hourly.idxmax()}:00")

print("\n3. GEOGRAPHIC:")
print(f"   - UK = {uk_pct:.0f}% of revenue")
print(f"   - International growth opportunity: {100-uk_pct:.0f}%")

print("\n4. RETENTION:")
print(f"   - Month 1 retention: {avg_ret[1]:.1%}")
print(f"   - Month 6 retention: {avg_ret[6]:.1%}" if 6 in avg_ret.index else "")

print("\n" + "=" * 50)
print("RECOMMENDATIONS")
print("=" * 50)
print("\n- Focus on Champions with VIP treatment/early access")
print("- Win-back campaigns for At Risk segment")
print("- Explore international markets (EIRE, Netherlands show high per-customer value)")
print("- Schedule promotions on Thursdays around noon")
